# 挑戰：分析關於資料科學的文本

在這個範例中，我們來做一個涵蓋傳統資料科學流程所有步驟的簡單練習。你不必寫任何程式碼，只需點擊下面的儲存格執行它們並觀察結果。作為挑戰，鼓勵你使用不同的資料來嘗試這段程式碼。

## 目標

在本課程中，我們已經討論了與資料科學相關的不同概念。讓我們嘗試透過做一些<strong>文本探勘</strong>來發掘更多相關概念。我們將從一段關於資料科學的文本開始，從中萃取關鍵詞，然後嘗試將結果視覺化。

作為文本，我將使用維基百科關於資料科學的頁面：


In [ ]:
url = 'https://en.wikipedia.org/wiki/Data_science'

## 第一步：取得資料

每個資料科學流程的第一步都是取得資料。我們將使用 `requests` 函式庫來做到這點：


In [ ]:
import requests

# Define a custom header.
headers = {
    'User-Agent': 'DataScienceChallenge/1.0 (myemail@gmail.com)'
}

# Pass the headers into the get request
response = requests.get(url, headers=headers)

if response.status_code == 200:
    text = response.content.decode('utf-8')
    print(text[:1000])
else:
    print(f"Error: {response.status_code}")

## 步驟 2：轉換資料

下一步是將資料轉換成適合處理的形式。以我們的情境來說，我們已經從網頁下載了 HTML 原始碼，而我們需要將它轉換成純文字。

有許多方法可以做到這件事。我們將使用 [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/)，一個流行的 Python HTML 解析函式庫。BeautifulSoup 讓我們可以針對特定的 HTML 元素處理，因此我們能專注於維基百科的主要文章內容，並減少一些導覽選單、側欄、頁尾以及其他無關內容（雖然某些樣板文字仍可能殘留）。


首先，我們需要安裝用於 HTML 解析的 BeautifulSoup 函式庫：


In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup

# Parse the HTML content
soup = BeautifulSoup(text, 'html.parser')

# Extract only the main article content from Wikipedia
# Wikipedia uses 'mw-parser-output' class for the main article content
content = soup.find('div', class_='mw-parser-output')

def clean_wikipedia_content(content_node):
    """Remove common non-article elements from a Wikipedia content node."""
    # Strip jump links, navboxes, reference lists/superscripts, edit sections, TOC, sidebars, etc.
    selectors = [
        '.mw-jump-link',
        '.navbox',
        '.reflist',
        'sup.reference',
        '.mw-editsection',
        '.hatnote',
        '.metadata',
        '.infobox',
        '#toc',
        '.toc',
        '.sidebar',
    ]
    for selector in selectors:
        for el in content_node.select(selector):
            el.decompose()

if content:
    # Clean the content node to better approximate article text only.
    clean_wikipedia_content(content)
    text = content.get_text(separator=' ', strip=True)
    print(text[:1000])
else:
    print("Could not find main content. Using full page text.")
    text = soup.get_text(separator=' ', strip=True)
    print(text[:1000])

## 第三步：獲取洞察

最重要的步驟是將資料轉換成我們能夠從中獲取洞察的某種形式。在我們的案例中，我們想從文本中擷取關鍵字，並查看哪些關鍵字更有意義。

我們將使用一個名為 [RAKE](https://github.com/aneesha/RAKE) 的 Python 函式庫來進行關鍵字擷取。首先，讓我們安裝這個函式庫，以防尚未安裝： 


In [ ]:
import sys
!{sys.executable} -m pip install nlp_rake

主要功能來自 `Rake` 物件，我們可以使用一些參數來自訂。就我們來說，我們會設定關鍵字的最小長度為 5 個字元，關鍵字在文件中的最小出現次數為 3，關鍵字中最大詞數為 2。歡迎自由嘗試其他數值並觀察結果。


In [ ]:
import nlp_rake
extractor = nlp_rake.Rake(max_words=2,min_freq=3,min_chars=5)
res = extractor.apply(text)
res


我們獲得了一組詞條及其相關的重要程度。如你所見，一些最相關的領域，例如機器學習和大數據，位於清單的前列。

## 第四步：結果視覺化

人們通常能更好地解讀視覺化的資料。因此，將資料視覺化以便獲得一些洞見常常是有意義的。我們可以使用 Python 的 `matplotlib` 函式庫來繪製關鍵詞與其相關度的簡單分佈圖：


In [ ]:
import matplotlib.pyplot as plt

def plot(pair_list):
    k,v = zip(*pair_list)
    plt.bar(range(len(k)),v)
    plt.xticks(range(len(k)),k,rotation='vertical')
    plt.show()

plot(res)

不過，有一種更好的方式來視覺化詞頻——使用 **文字雲（Word Cloud）**。我們需要安裝另一個函式庫來從我們的關鍵字列表繪製文字雲。


In [ ]:
!{sys.executable} -m pip install wordcloud

`WordCloud` 物件負責接收原始文本或預先計算的詞彙頻率清單，並回傳一張圖像，該圖像可以用 `matplotlib` 進行顯示：


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(background_color='white',width=800,height=600)
plt.figure(figsize=(15,7))
plt.imshow(wc.generate_from_frequencies({ k:v for k,v in res }))

我們也可以將原始文字傳入 `WordCloud` — 讓我們看看是否能得到類似的結果：


In [ ]:
plt.figure(figsize=(15,7))
plt.imshow(wc.generate(text))

In [ ]:
wc.generate(text).to_file('images/ds_wordcloud.png')

你現在可以看到詞雲看起來更令人印象深刻，但它也包含了很多噪音（例如不相關的詞如 `Retrieved on`）。此外，我們得到的由兩個詞組成的關鍵字較少，例如 *data scientist* 或 *computer science*。這是因為 RAKE 演算法在從文本中選取好的關鍵字時表現更佳。這個例子說明了資料預處理和清理的重要性，因為最後清晰的圖像將讓我們做出更好的決策。

在這個練習中，我們經歷了一個簡單的過程，從維基百科文本中提取一些意義，以關鍵字和詞雲的形式呈現。這個例子相當簡單，但它很好地展現了資料科學家在處理數據時會採取的所有典型步驟，從資料獲取到視覺化。

在我們的課程中，我們將詳細討論所有這些步驟。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
此文件已使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們努力追求準確性，但請注意自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應視為權威來源。對於關鍵資訊，建議採用專業人工翻譯。我們不對因使用此翻譯所產生的任何誤解或誤譯承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
